# DE2 - Bayern

In [1]:
import pandas as pd
import zipfile
from glob import glob
from pathlib import Path

from tqdm import tqdm
import pyproj
from camelsp import get_metadata

from camels_de1h import Station1h, get_input_path, get_nuts_id_from_provider_id

## Extract data

There are two .csv files for each station and variable: 
- start of measuring period - 2024-31-12
- 2025-01-01 - 2025-10-31
- 2025-11-01 - 2025-11-XX
- 2025-11-2X (day of data download)

We only use the first two files as we do not need the last few days.

In [4]:
input_path = get_input_path("DE2")

if not (input_path / "fluesse-wasserstand").exists() and not (input_path / "Bayern" / "fluesse-abfluss").exists():
    # Unzip the Bayern.zip file
    with zipfile.ZipFile(input_path / "Bayern.zip", "r") as zip_ref:
        zip_ref.extractall(input_path)

    # only use the two files for start-2025 and 2025-October
    remove_files = glob(str(input_path / "fluesse-wasserstand" / f"*_*.11.2025_ezw_0.csv"))
    remove_files.extend(glob(str(input_path / "fluesse-abfluss" / f"*_*.11.2025_ezw_0.csv")))
    for remove_file in remove_files:
        Path(remove_file).unlink()
else:
    print("Data already extracted.")

## Parse data

**Timezone: MEZ** -> convert to UTC+0

In [29]:
def parse_station_data(id: str, variable: str, aggregate_hourly: bool, to_utc: bool) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Parse station data with flexible header handling.
    
    """
    if variable not in ["abfluss", "wasserstand"]:
        raise ValueError("Invalid variable. Use 'abfluss' or 'wasserstand'.")
    
    # Try to find files, return None if not found
    file_path1_list = list((input_path / f"fluesse-{variable}").glob(f"{id}_beginn_bis_*.csv"))
    file_path1 = file_path1_list[0] if file_path1_list else None
    
    file_path2_list = list((input_path / f"fluesse-{variable}").glob(f"{id}_01.01.2025_31.10.2025_ezw_0.csv"))
    file_path2 = file_path2_list[0] if file_path2_list else None
    
    column_name = "discharge_vol_obs" if variable == "abfluss" else "water_level_obs"

    def parse_file(file_path):
        if file_path is None:
            return None, None
            
        try:
            header_lines = []
            data_start_line = 0
            
            # First pass: read header and check for "no data" message
            with open(file_path, 'r', encoding='utf-8-sig') as file:
                # Read lines until we find the "Datum" line
                while True:
                    line = file.readline().strip()
                    data_start_line += 1
                    
                    if not line:  # Empty line
                        continue
                    
                    if line.startswith("Datum"):
                        # Check next line for "no data" message
                        next_line = file.readline().strip()
                        if "Es liegen keine Daten für den Zeitraum" in next_line:
                            # Convert header to DataFrame before returning
                            header = pd.DataFrame(header_lines)
                            header = header.T
                            header.columns = header.iloc[0]
                            header = header[1:].reset_index(drop=True)
                            if "Pegelnullpunktshöhe" not in header.columns:
                                header["Pegelnullpunktshöhe"] = None
                            return header, None
                        break
                        
                    # Special handling for coordinates line
                    if "Ostwert" in line:
                        parts = line.split(";")
                        header_lines.append(["Ostwert", parts[1]])
                        header_lines.append(["Nordwert", parts[3]])
                        header_lines.append(["Koordinatensystem", parts[4].strip('"')])
                        continue
                        
                    # Normal header line
                    parts = line.split(";")
                    header_lines.append([parts[0].strip('":'), parts[1].strip('"')])

            # Validate required header fields
            required_fields = ["Quelle", "Messstellen-Name", "Messstellen-Nr.", "Gewässer"]
            header_keys = [line[0] for line in header_lines]
            missing_fields = [field for field in required_fields if field not in header_keys]
            if missing_fields:
                raise ValueError(f"Missing required header fields: {missing_fields}")

            # Convert header to DataFrame
            header = pd.DataFrame(header_lines)
            header = header.T
            header.columns = header.iloc[0]
            header = header[1:].reset_index(drop=True)

            # Pegelnullpunktshöhe is not always in header
            if "Pegelnullpunktshöhe" not in header.columns:
                header["Pegelnullpunktshöhe"] = None
                
            # Parse data
            data = pd.read_csv(file_path, delimiter=";", skiprows=data_start_line, header=None,
                            usecols=[0, 1], names=["date", column_name], decimal=",",
                            parse_dates=["date"], date_format="%Y-%m-%d %H:%M")
            return header, data
            
        except FileNotFoundError:
            return None, None
        except Exception as e:
            print(f"Error parsing file {file_path}: {str(e)}")
            return None, None

    header1, data1 = parse_file(file_path1)
    header2, data2 = parse_file(file_path2)

    # Check if any data was found
    if data1 is None and data2 is None:
        return None, None

    # Combine data
    data = pd.concat([data1, data2] if data1 is not None and data2 is not None 
                     else [d for d in [data1, data2] if d is not None], 
                     axis=0).reset_index(drop=True)

    if to_utc:
        data["date"] = data.set_index("date").index.tz_localize("Etc/GMT-1").tz_convert("UTC")

    if aggregate_hourly:
        data = data.set_index("date").resample("h", label="right", closed="right").mean().reset_index()

    # Use the non-None header, preferring header1
    header = header1 if header1 is not None else header2

    return header, data

parse_station_data("10026301", "wasserstand", aggregate_hourly=True, to_utc=True)

(0                                             Quelle  Datenbankabfrage  \
 0  Bayerisches Landesamt für Umwelt, www.gkd.baye...  15.10.2025 00:00   
 
 0 Zeitbezug   Messstellen-Name Messstellen-Nr. Gewässer Ostwert Nordwert  \
 0       MEZ  Neu-Ulm, Bad Held        10026301    Donau  573054  5360044   
 
 0      Koordinatensystem      Pegelnullpunktshöhe  
 0  ETRS89 / UTM Zone 32N  464,86 m NHN (DHHN2016)  ,
                             date  water_level_obs
 0      1981-10-31 23:00:00+00:00            315.0
 1      1981-11-01 00:00:00+00:00            315.5
 2      1981-11-01 01:00:00+00:00            316.0
 3      1981-11-01 02:00:00+00:00            317.0
 4      1981-11-01 03:00:00+00:00            317.0
 ...                          ...              ...
 382964 2025-07-09 19:00:00+00:00              NaN
 382965 2025-07-09 20:00:00+00:00              NaN
 382966 2025-07-09 21:00:00+00:00              NaN
 382967 2025-07-09 22:00:00+00:00              NaN
 382968 2025-07-09 23:00

The header does not contain area information, I researched that online (where available: hnd.bayern.de/pegel) and created a mapping here.

In [30]:
area_mapping = {
    "11400505": 317.60,
    "13191030": 93.9,
    "13227000": 110.1,
    "13325207": 234.9,
    "13327655": 95.14,
    "13406555": 111.4,
    "13455020": 78.5,
    "14106505": 915.0,
    "15207055": 2530.4,
    "16163000": 106.38,
    "16345007": 33.9,
    "16395005": 19.1,
    "16425003": 96.2,
    "18269055": 3.64,
    "18460403": 21.7,
    "18624003": 39.5,
    "24142855": 8.3,
    "24213101": 9.24,
    "10026293": 7616.6,
    "11405001": 2112.0,
    "11415008": 75.4,
    "12393201": 0.01,
    "12483000": 101.0,
    "12483100": None,
    "13323955": 38.63,
    "13327855": 72.5,
    "16005703": 2865.0,
    "16408506": 1051.0,
    "16495000":0.01,
    "18206003": 780.0,
    "18209001": 1104.0,
    "18284606": 0.01,
    "18408201": 2217.0,
    "18463006": 98.9,
    "24118355": 10.3
}

Use the function to parse data for all stations and save raw metadata from the header.

In [45]:
def clean_gauge_name(gauge_name: str, water_body_name: str) -> str:
    """Function to clean gauge names by removing river part if it matches water body name (already fixed in CAMELS-DE v1.1.0)."""
    if "_" in gauge_name:
        gauge_part, river_part = gauge_name.rsplit("_", 1)

        if river_part in water_body_name:
            gauge_name = gauge_part.strip()
        else:
            print(f"Warning: Gauge name '{gauge_name}' does not match water body name '{water_body_name}'.")
    return gauge_name.strip()

In [ ]:
ids = list(sorted(set([f.split("/")[-1].split("_")[0] for f in glob(str(input_path / "fluesse-*" / "*_01.01*.csv"))])))

# get the metadata from CAMELS-DE daily
camelsp_meta_all = get_metadata()

# Read the old raw metadata from CAMELS-DE daily
meta_old_raw_all = pd.read_excel(input_path / "Stammdaten_Bayern.xlsx")

for id in tqdm(ids):
    # get the gauge_id from the nuts_mapping, add it if it does not exist
    gauge_id = get_nuts_id_from_provider_id(id, "DE2", add_missing=True)

    # parse metadata and data for the station
    header_q, data_q = parse_station_data(id, "abfluss", aggregate_hourly=True, to_utc=True)
    header_w, data_w = parse_station_data(id, "wasserstand", aggregate_hourly=True, to_utc=True)

    if header_q is None and data_q is None:
        header = header_w
        data = data_w
    elif header_w is None and data_w is None:
        header = header_q
        data = data_q
    else:
        # Check if the headers are the same (ignoring Datenbankabfrage timestamp)
        header_q_check = header_q.drop(columns="Datenbankabfrage", errors='ignore')
        header_w_check = header_w.drop(columns="Datenbankabfrage", errors='ignore')
        
        if not header_q_check.equals(header_w_check):
            # Find differences
            diff_cols = []
            for col in set(header_q_check.columns) | set(header_w_check.columns):
                if col not in header_q_check.columns:
                    diff_cols.append(f"{col} (only in wasserstand)")
                elif col not in header_w_check.columns:
                    diff_cols.append(f"{col} (only in abfluss)")
                elif header_q_check[col].values[0] != header_w_check[col].values[0]:
                    diff_cols.append(f"{col}: '{header_q_check[col].values[0]}' vs '{header_w_check[col].values[0]}'")
            
            print(f"Header of station {id} differs in: {', '.join(diff_cols)}")
            
        header = header_q

        # Combine the data
        data = pd.merge(data_q, data_w, on="date", how="outer")

        # Sort the data by date
        data = data.sort_values("date").reset_index(drop=True)

    # get the metadata of the station
    camelsp_meta = camelsp_meta_all[camelsp_meta_all["camels_id"] == gauge_id]
    meta_old_raw = meta_old_raw_all[meta_old_raw_all["Stationsnummer"].astype(str) == id]

    if not camelsp_meta.empty:
        # fix gauge names (also fixed in CAMELS-DE v1.1.0)
        gauge_name = clean_gauge_name(
            camelsp_meta["gauge_name"].values[0],
            camelsp_meta["waterbody_name"].values[0]
        )

        # set metadata values from camelsp metadata
        area_metadata = camelsp_meta["area"].values[0]
        # Clean area metadata - replace values with "  " with NaN
        if isinstance(area_metadata, str) and "  " in area_metadata:
            area_metadata = float('nan')
        
        station_meta = {
            "gauge_id": camelsp_meta["camels_id"].values[0],
            "provider_id": camelsp_meta["provider_id"].values[0],
            "gauge_name": gauge_name,
            "waterbody_name": camelsp_meta["waterbody_name"].values[0].strip(),
            "federal_state": camelsp_meta["federal_state"].values[0],
            "gauge_lon": camelsp_meta["lon"].values[0],
            "gauge_lat": camelsp_meta["lat"].values[0],
            "gauge_easting": camelsp_meta["x"].values[0],
            "gauge_northing": camelsp_meta["y"].values[0],
            "gauge_elev_metadata": camelsp_meta["gauge_elevation"].values[0],
            "area_metadata": area_metadata,
            "part_of_camelsp": True,
        }
    elif not meta_old_raw.empty:
        # fix gauge names (also fixed in CAMELS-DE v1.1.0)
        gauge_name = clean_gauge_name(
            meta_old_raw["Stationsname"].values[0],
            meta_old_raw["Gewässer (Name|Nummer)"].values[0]
        )
        
        # parse the location
        easting, northing = meta_old_raw["Ostwert"].values[0], meta_old_raw["Nordwert"].values[0]

        transformer = pyproj.Transformer.from_crs("epsg:25832", "epsg:4326", always_xy=True)
        lon, lat = transformer.transform(easting, northing)

        # from EPSG:31466 to EPSG:3035
        transformer = pyproj.Transformer.from_crs("epsg:25832", "epsg:3035", always_xy=True)
        easting, northing = transformer.transform(easting, northing)

        # Clean area metadata - replace values with "  " with NaN
        area_metadata = meta_old_raw["EZG km²"].values[0]
        if isinstance(area_metadata, str) and "  " in area_metadata:
            area_metadata = float('nan')

        # if the station is not in the camelsp metadata, use the raw metadata of camelsp / CAMELS-DE
        station_meta = {
            "gauge_id": gauge_id,
            "provider_id": id,
            "gauge_name": gauge_name,
            "waterbody_name": meta_old_raw["Gewässer (Name|Nummer)"].values[0].strip(),
            "federal_state": "Bayern",
            "gauge_lon": lon,
            "gauge_lat": lat,
            "gauge_easting": easting,
            "gauge_northing": northing,
            "gauge_elev_metadata": meta_old_raw["PNP"].values[0],
            "area_metadata": area_metadata,
            "part_of_camelsp": False,
        }
    else:
        # if the station is also not in the excel metadata from CAMELS-DE, use the raw metadata from the .csv files
        easting, northing = header["Ostwert"].values[0], header["Nordwert"].values[0]

        transformer = pyproj.Transformer.from_crs("epsg:25832", "epsg:4326", always_xy=True)
        lon, lat = transformer.transform(easting, northing)

        # from EPSG:31466 to EPSG:3035
        transformer = pyproj.Transformer.from_crs("epsg:25832", "epsg:3035", always_xy=True)
        easting, northing = transformer.transform(easting, northing)

        # elevation
        if header["Pegelnullpunktshöhe"].values[0] is not None:
            elevation = float(header["Pegelnullpunktshöhe"].values[0].split(" ")[0].replace(",", "."))
        else:
            elevation = None

        station_meta = {
            "gauge_id": gauge_id,
            "provider_id": id,
            "gauge_name": header["Messstellen-Name"].values[0],
            "waterbody_name": header["Gewässer"].values[0],
            "federal_state": "Bayern",
            "gauge_lon": lon,
            "gauge_lat": lat,
            "gauge_easting": easting,
            "gauge_northing": northing,
            "gauge_elev_metadata": elevation,
            "area_metadata": area_mapping.get(id, None),
            "part_of_camelsp": False,
        }
        
    # initialize the station
    station = Station1h(gauge_id)

    # save time series data
    station.save_data(data)

    # save metadata
    station.save_metadata(**station_meta)
    
    # save raw metadata
    station.save_raw_metadata(header)

  0%|          | 1/549 [00:02<23:41,  2.59s/it]

Header of station 10026301 differs in: Messstellen-Name: 'Neu Ulm, Bad Held' vs 'Neu-Ulm, Bad Held'


 17%|█▋        | 93/549 [06:10<34:23,  4.52s/it]

 28%|██▊       | 151/549 [09:22<29:25,  4.44s/it]

 63%|██████▎   | 346/549 [21:28<11:14,  3.32s/it]

100%|██████████| 549/549 [34:38<00:00,  3.79s/it]
